In [71]:
import os
import glob
import random
import pandas as pd
import numpy as np
import torch  
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from numba import njit, float64, int64, uint64,types
from numba.typed import Dict
from tqdm import tqdm
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [72]:
# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# 실제 데이터가 있는 경로로 수정하세요
BASE_PATH = "C:/Users/user/Desktop/IDS_masters/Car_Hacking_Challenge_Dataset_rev20Mar2021/0_Preliminary/0_Training/0_Training_test"
PT_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/training_dataset_yj.pt"
CSV_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/training_dataset_yj.csv"
WINDOW_SIZE = 128
STRIDE = 64  # 50% Overlap

# 공격 라벨 정의
ATTACK_LABELS = {
    "Normal": 0,
    "Flooding": 1,
    "Fuzzing": 2,
    "Replay": 3,
    "Spoofing": 4
}
LABEL_MAP = {
    "Normal": 0,
    "Flooding": 1,   # 원본 명칭
    "DoS": 1,        # 혹시 나중에 DoS라는 문자열도 들어오면 같이 1로 처리
    "Fuzzing": 2,
    "Replay": 3,
    "Spoofing": 4,
}

FEATURE_NAMES = [
    "IAT", "Is_Zero", "Payload_Ent", "Complexity", 
    "Ham_Rate", "Freq", "Continuity", "Diff_Ent", "ID_Ent",
    "Freq_Fast_Z", "Jit_Fast_Z","?","@","0","ㅎㅎㅎ"
]



In [73]:
@njit
def popcount64(x):
    c = 0
    v = int64(x)
    while v:
        v &= v - int64(1)
        c += 1
    return c

@njit
def pack_payload_u64(row):
    v = uint64(0)
    for i in range(8):
        v |= uint64(row[i]) << (i * 8)
    return v

@njit
def update_ema_z(val, cid, ema_map, sq_ema_map, alpha):
    if cid not in ema_map:
        ema_map[cid] = float64(val)
        sq_ema_map[cid] = float64(val ** 2)
        return 0.0
    mean = ema_map[cid]
    sq_mean = sq_ema_map[cid]
    var = sq_mean - (mean ** 2)
    if var < 0: var = 0.0
    std = np.sqrt(var)
    z = 0.0
    if std > 1e-9:
        z = (val - mean) / std
        if z > 5.0: z = 5.0
        elif z < -5.0: z = -5.0
    ema_map[cid] = (1.0 - alpha) * mean + alpha * val
    sq_ema_map[cid] = (1.0 - alpha) * sq_mean + alpha * (val ** 2)
    return z

@njit(fastmath=True)
def calculate_features_15_numba(timestamps, can_ids, payloads):
    n = len(timestamps)
    features = np.zeros((n, 15), dtype=np.float64)
    
    # 상태 관리
    last_time_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64)
    last_id_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_iat_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_global_index_map = Dict.empty(key_type=types.int64, value_type=types.int64)
    id_ham_ema = Dict.empty(key_type=types.int64, value_type=types.float64)
    
    # Anchor & Warm-up (128개 윈도우 고려)
    WARM_UP_LIMIT = 4000 
    anchor_iat_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    anchor_gap_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_normal_time_map = Dict.empty(key_type=types.int64, value_type=types.float64)

    # Z-Score & Global
    ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    ema_jit = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_jit = Dict.empty(key_type=types.int64, value_type=types.float64)
    ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)

    G_KEY = np.int64(-1)
    alpha_slow = 0.001
    alpha_ham = 0.05
    eps = 1e-9
    prev_global_time = timestamps[0]

    for i in range(n):
        # 128개 윈도우 기준 로컬 빈도 초기화
        if (i % 128) == 0:
            last_id_map.clear()
            
        ts = timestamps[i]
        cid = can_ids[i]
        row = payloads[i]
        if np.isnan(ts): ts = prev_global_time

        # 1. 물리량
        curr_iat = max(0.0, ts - last_time_map[cid]) if cid in last_time_map else 0.001
        curr_packet_gap = float64(i - last_global_index_map[cid]) if cid in last_global_index_map else 100.0
        curr_freq = 1.0 / (curr_iat + eps)
        curr_jit = np.abs(curr_iat - last_iat_map[cid]) if cid in last_iat_map else 0.0

       # 2. [핵심] Warm-up: 시간 주기와 패킷 개수 주기를 동시에 학습/고정
        if i < WARM_UP_LIMIT:
            if cid not in anchor_iat_map:
                anchor_iat_map[cid] = curr_iat
                anchor_gap_map[cid] = curr_packet_gap
            else:
                # 4000개까지는 부드럽게 평균을 쌓음
                anchor_iat_map[cid] = 0.99 * anchor_iat_map[cid] + 0.01 * curr_iat
                anchor_gap_map[cid] = 0.99 * anchor_gap_map[cid] + 0.01 * curr_packet_gap
                
        # 3. 시간/패킷 기반 스푸핑 분석 (Clipping)
        iat_ratio,packet_gap_ratio, phase_offset = 1.0,1.0, 0.0
        if cid in anchor_iat_map:
            # 3-1. 시간 기반 분석
            base_iat = anchor_iat_map[cid]
            if base_iat > 1e-7:
                iat_ratio = min(curr_iat / base_iat, 2.0)
                diff = ts - (last_normal_time_map[cid] + base_iat) if cid in last_normal_time_map else 0.0
                phase_offset = max(min(diff / base_iat, 1.0), -1.0)
            
            # 3-2. [신규] 패킷 개수 기반 분석 (사용자 제안 로직)
            base_gap = anchor_gap_map[cid]
            if base_gap > 0.5:
                # 평소 100개 뒤에 나오던 게 5개 뒤에 나오면 0.05가 됨
                packet_gap_ratio = min(curr_packet_gap / base_gap, 2.0)

            # 리듬 업데이트 (시간 기준)
            if abs(phase_offset) < 0.2:
                last_normal_time_map[cid] = ts
        else:
            last_normal_time_map[cid] = ts

        # 4. 데이터 내용 기반 (Hamming & Entropy)
        cur_bytes = pack_payload_u64(row)
        rel_change = 0.0
        if cid in last_payload_map:
            h_dist = float64(popcount64(cur_bytes ^ last_payload_map[cid]))
            avg_h = id_ham_ema.get(cid, h_dist)
            rel_change = h_dist / (avg_h + 0.1)
            id_ham_ema[cid] = (1.0 - alpha_ham) * avg_h + alpha_ham * h_dist
        last_payload_map[cid] = cur_bytes

        # Payload Entropy
        p_counts = np.zeros(256, dtype=np.int64)
        for b in row: p_counts[b] += 1
        ent = 0.0
        for c in p_counts:
            if c > 0:
                p = c / 8.0
                ent -= p * np.log(p)

        # Diff Entropy
        d_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
        for b_idx in range(7):
            d = (int64(row[b_idx+1]) - int64(row[b_idx])) % 256
            d_counts[d] = d_counts.get(d, 0.0) + 1.0
        d_ent = 0.0
        for dv in d_counts:
            pk = d_counts[dv] / 7.0
            d_ent -= pk * np.log(pk + 1e-9)

        # 윈도우 ID 엔트로피 (최근 128개 패킷 대상)
        wi_ent = 0.0
        if i >= 127:
            win_id_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
            for k in range(i-127, i+1):
                wid = can_ids[k]
                win_id_counts[wid] = win_id_counts.get(wid, 0.0) + 1.0
            for wid in win_id_counts:
                pk = win_id_counts[wid] / 128.0
                wi_ent -= pk * np.log(pk + 1e-9)

        # 5. 피처 할당
        features[i, 0] = np.log1p(curr_iat * 1000.0) / 7.0
        features[i, 1] = 1.0 if cid == 0 else 0.0
        features[i, 2] = ent / 2.1
        features[i, 3] = np.log1p(ent * rel_change)
        features[i, 4] = np.log1p(rel_change / (curr_iat + eps)) / 10.0
        cnt = last_id_map.get(cid, 0.0) + 1.0
        last_id_map[cid] = cnt
        features[i, 5] = cnt / 128.0
        features[i, 6] = np.log1p(rel_change) / 5.0
        features[i, 7] = d_ent / 1.94
        features[i, 8] = wi_ent / 4.85
        features[i, 9] = update_ema_z(curr_freq, cid, ema_freq, sq_ema_freq, alpha_slow)
        features[i, 10] = update_ema_z(curr_iat, cid, ema_jit, sq_ema_jit, alpha_slow)
        features[i, 11] = update_ema_z(curr_freq, G_KEY, ema_global, sq_ema_global, alpha_slow)
        features[i, 12] = iat_ratio
        features[i, 13] = phase_offset
        features[i, 14] = packet_gap_ratio

        last_time_map[cid] = ts
        last_iat_map[cid] = curr_iat
        last_global_index_map[cid] = i
        prev_global_time = ts

    return features

In [74]:
def make_windows_from_stream(features, labels, window_size=128, stride=64):
    """
    features: (N, F)
    labels:   (N,)  또는 (N, ) per packet label
    return:
      Xw: (Nwin, F, L)
      yw: (Nwin, L)
    """
    N, F = features.shape
    L = window_size

    nwin = 1 + (N - L) // stride
    Xw = np.zeros((nwin, F, L), dtype=np.float32)
    yw = np.zeros((nwin, L), dtype=np.int64)

    w = 0
    for start in range(0, N - L + 1, stride):
        end = start + L
        # (L, F) -> (F, L)
        Xw[w] = features[start:end].T.astype(np.float32)
        yw[w] = labels[start:end].astype(np.int64)
        w += 1

    return Xw, yw

In [75]:
# ==========================================
# 2. 헬퍼 함수 (ID 파싱, Payload 파싱)
# ==========================================
def parse_id(id_val):
    if isinstance(id_val, str):
        try:
            return int(id_val, 16)
        except:
            return 0
    return int(id_val)

def parse_payload_str(s, max_len=8):
    """'00 00 A1 ...' 형태의 문자열을 길이 8의 리스트로 변환"""
    parts = str(s).split()
    vals = []
    for p in parts:
        if p != "":
            try:
                vals.append(int(p, 16))
            except:
                pass
    
    if len(vals) < max_len:
        vals += [0] * (max_len - len(vals))
    return vals[:max_len]

In [76]:
def main():
    # 1. CSV 파일 목록
    csv_files = glob.glob(os.path.join(BASE_PATH, "*.csv"))
    if not csv_files:
        print(f"[ERROR] 해당 경로에 CSV 파일이 없습니다: {BASE_PATH}")
        return

    print(f"[INFO] 발견된 파일: {len(csv_files)}개")
    for f in csv_files:
        print("   -", os.path.basename(f))

    # 2. CSV 통합
    df_list = []
    for file in csv_files:
        print(f"[READING] {os.path.basename(file)} 읽는 중...")
        temp_df = pd.read_csv(file, header=0)
        df_list.append(temp_df)

    full_df = pd.concat(df_list, axis=0, ignore_index=True)
    print(f"[INFO] 통합 완료. 총 패킷 수: {len(full_df)}")

    # 3. 라벨 정리 (Flooding → DoS 이름 통일)
    full_df["SubClass"] = full_df["SubClass"].astype(str).str.strip()
    full_df["SubClass"] = full_df["SubClass"].replace("Flooding", "DoS")

    # 패킷 단위 문자열 라벨
    raw_labels = full_df["SubClass"].astype(str).values

    # 4. 피처 계산에 필요한 컬럼 → numpy
    timestamps = full_df["Timestamp"].astype(np.float64).to_numpy()
    can_ids    = full_df["Arbitration_ID"].apply(parse_id).astype(np.int64).to_numpy()
    payload_array = np.vstack(
        full_df["Data"].apply(parse_payload_str).values
    ).astype(np.uint8)

    print("[INFO] 통합 데이터 피처 계산 중...")
    
    # all_features.shape = (패킷 수, 9)

    # 5. ===== 패킷 레벨 CSV 저장 =====
    packet_labels_int = np.vectorize(LABEL_MAP.get)(raw_labels).astype(np.int64)

    feat_stream = calculate_features_15_numba(timestamps, can_ids, payload_array)
    X_np, y_np = make_windows_from_stream(feat_stream, packet_labels_int, window_size=128, stride=64)

    df_packet = pd.DataFrame(feat_stream, columns=FEATURE_NAMES)
    df_packet["Label_Int"] = packet_labels_int
    df_packet["Label_Str"] = raw_labels
    df_packet.to_csv(CSV_SAVE_PATH, index=False)
    print(f"[DONE] .csv 패킷 단위 저장 완료: {CSV_SAVE_PATH}")
    print(f"       형태: {df_packet.shape} (행: 패킷 수, 열: 특징+라벨)")

    np.savez(
    "C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_test_0212_908.npz",
    X=X_np.astype(np.float32),
    y=y_np.astype(np.int64)
    )

    print(f" Saved dataset")

if __name__ == "__main__":
    main()

[INFO] 발견된 파일: 4개
   - Pre_train_D_1.csv
   - Pre_train_D_2.csv
   - Pre_train_S_1.csv
   - Pre_train_S_2.csv
[READING] Pre_train_D_1.csv 읽는 중...
[READING] Pre_train_D_2.csv 읽는 중...
[READING] Pre_train_S_1.csv 읽는 중...
[READING] Pre_train_S_2.csv 읽는 중...
[INFO] 통합 완료. 총 패킷 수: 3312119
[INFO] 통합 데이터 피처 계산 중...
[DONE] .csv 패킷 단위 저장 완료: C:/Users/user/Desktop/IDS_masters/training_dataset_yj.csv
       형태: (3312119, 17) (행: 패킷 수, 열: 특징+라벨)
 Saved dataset


In [77]:
import numpy as np

Path = "C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_test_0212_908.npz"

data = np.load(Path)

X = data["X"]
y = data["y"]

print(f"x shape: {X.shape}")
print(f"y shape: {y.shape}")

unique, counts = np.unique(y, return_counts=True)
print(unique)
print(counts)

uniq_dict = dict(zip(unique, counts))
print(f"클래스 별 데이터 수: {uniq_dict}")

x shape: (51750, 15, 128)
y shape: (51750, 128)
[0 1 2 3 4]
[6025184  308360  179758   95186   15512]
클래스 별 데이터 수: {np.int64(0): np.int64(6025184), np.int64(1): np.int64(308360), np.int64(2): np.int64(179758), np.int64(3): np.int64(95186), np.int64(4): np.int64(15512)}
